# Step 4 — Multi-Sensor Track Fusion (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_3/lidar/track_*.json`, `output/step_3/radar/track_*.json` (tuple format) |
| | `output/step_3/camera/track_*.json` (dict format, from Step 3.3) |
| **Outputs** | `output/step_4/fused_tracks_all.csv` — one row per fused-track point (fast, primary output) |
| | `output/step_4/fused/track_<id>.json` — one file per fused track (now feasible — thousands, not millions) |
| | `output/step_4/fusion_summary.csv` |
| **Used by** | Step 5 (TTC estimation) |

---

### The real cause of the 22-hour runtime

`active_fused_objects = detections` at the end of each loop carried forward **every raw detection** from the current frame (~10,000+ points, dominated by radar) as the matching pool for the next frame. The nested loop then ran roughly 100 million scalar `euclidean()` calls per frame. That's the 196.72s/iteration you measured — not primarily the file writes, though writing 1M+ individual JSON files made it worse.

### The fix: fuse at the TRACK level, not the raw-point level

Steps 3.1/3.2/3.3 already turned millions of raw points into a few thousand clean, deduplicated tracks. This notebook now fuses those tracks — a problem that's orders of magnitude smaller and doesn't need any raw point data at all.

### Also fixed
- **Format compatibility** — loaders now correctly parse Step 3.1/3.2's tuple format and Step 3.3's dict/trajectory format.
- **Converted from Colab (Drive mount + zip) to local, config.py-based** — consistent with the rest of the pipeline.
- **Proper one-shot Hungarian assignment per frame** instead of nested nearest-neighbor nested loops.
- **Within-frame sensor merging** — if LiDAR and radar both observe the same real object in the same frame, they're merged into one observation (averaged position, sensors recorded) before being matched against existing fused tracks. This is the actual "fusion" step that was largely absent before.
- **Track eviction** — same `MAX_MISSED_FRAMES` pattern as Steps 3.1–3.3.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP3_DIR, STEP4_DIR

LIDAR_TRACKS_DIR  = STEP3_DIR / "lidar"
RADAR_TRACKS_DIR  = STEP3_DIR / "radar"
CAMERA_TRACKS_DIR = STEP3_DIR / "camera"
FUSED_OUT_DIR      = STEP4_DIR / "fused"
FUSED_OUT_DIR.mkdir(parents=True, exist_ok=True)

for p, name in [(LIDAR_TRACKS_DIR, "Step 3.1"), (RADAR_TRACKS_DIR, "Step 3.2"), (CAMERA_TRACKS_DIR, "Step 3.3")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} output not found at {p} — run that step first.")

print(f"✅ LIDAR_TRACKS_DIR : {LIDAR_TRACKS_DIR}")
print(f"✅ RADAR_TRACKS_DIR : {RADAR_TRACKS_DIR}")
print(f"✅ CAMERA_TRACKS_DIR: {CAMERA_TRACKS_DIR}")
print(f"✅ FUSED_OUT_DIR    : {FUSED_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ LIDAR_TRACKS_DIR : F:\Sensor fusion Research\output\step_3\lidar
✅ RADAR_TRACKS_DIR : F:\Sensor fusion Research\output\step_3\radar
✅ CAMERA_TRACKS_DIR: F:\Sensor fusion Research\output\step_3\camera
✅ FUSED_OUT_DIR    : F:\Sensor fusion Research\output\step_4\fused


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

INTRA_FRAME_MERGE_THRESH = 2.5   # metres — merge same-frame observations from different sensors this close together
FUSION_DIST_THRESHOLD    = 3.0   # metres — max distance to match a frame observation to an existing fused track
MAX_MISSED_FRAMES        = 3     # frames before a fused track is evicted

print(f"✅ INTRA_FRAME_MERGE_THRESH = {INTRA_FRAME_MERGE_THRESH}m")
print(f"✅ FUSION_DIST_THRESHOLD    = {FUSION_DIST_THRESHOLD}m")
print(f"✅ MAX_MISSED_FRAMES        = {MAX_MISSED_FRAMES}")

✅ INTRA_FRAME_MERGE_THRESH = 2.5m
✅ FUSION_DIST_THRESHOLD    = 3.0m
✅ MAX_MISSED_FRAMES        = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Loaders: convert each sensor's track format into a common
# per-sample observation list: {sensor, source_track_id, pos}
# ─────────────────────────────────────────────────────────────────

import json
from collections import defaultdict


def load_lidar_or_radar_tracks(track_dir, sensor_name):
    """Step 3.1/3.2 format: each track_*.json is a list of
    (sample_id, timestamp, [x,y,z]) tuples."""
    observations_by_sample = defaultdict(list)
    for file in track_dir.glob("track_*.json"):
        source_track_id = file.stem
        with open(file) as f:
            points = json.load(f)
        for sample_id, timestamp, pos in points:
            observations_by_sample[sample_id].append({
                "sensor": sensor_name,
                "source_track_id": source_track_id,
                "pos": pos
            })
    return observations_by_sample


def load_camera_tracks(track_dir):
    """Step 3.3 format: each track_*.json is a dict with a 'trajectory' list."""
    observations_by_sample = defaultdict(list)
    for file in track_dir.glob("track_*.json"):
        with open(file) as f:
            data = json.load(f)
        source_track_id = data["track_id"]
        for pt in data["trajectory"]:
            observations_by_sample[pt["sample_id"]].append({
                "sensor": "camera",
                "source_track_id": source_track_id,
                "pos": pt["pos"]
            })
    return observations_by_sample


lidar_by_sample  = load_lidar_or_radar_tracks(LIDAR_TRACKS_DIR, "lidar")
radar_by_sample  = load_lidar_or_radar_tracks(RADAR_TRACKS_DIR, "radar")
camera_by_sample = load_camera_tracks(CAMERA_TRACKS_DIR)

# Diagnostic: how many detections is each sensor actually contributing per frame on average?
for name, by_sample in [("lidar", lidar_by_sample), ("radar", radar_by_sample), ("camera", camera_by_sample)]:
    total_points = sum(len(v) for v in by_sample.values())
    n_samples_with_data = sum(1 for v in by_sample.values() if len(v) > 0)
    avg_per_frame = total_points / n_samples_with_data if n_samples_with_data > 0 else 0
    print(f"{name}: {total_points} total points, {n_samples_with_data} samples with data, "
          f"avg {avg_per_frame:.1f} detections/frame")

n_lidar  = sum(len(v) for v in lidar_by_sample.values())
n_radar  = sum(len(v) for v in radar_by_sample.values())
n_camera = sum(len(v) for v in camera_by_sample.values())

print(f"✅ Loaded {n_lidar} LiDAR track-points")
print(f"✅ Loaded {n_radar} Radar track-points")
print(f"✅ Loaded {n_camera} Camera track-points")
print(f"   Total: {n_lidar + n_radar + n_camera} (compare to the original's millions — this is tracks, not raw points)")

lidar: 40567 total points, 404 samples with data, avg 100.4 detections/frame
radar: 62266 total points, 398 samples with data, avg 156.4 detections/frame
camera: 4399 total points, 377 samples with data, avg 11.7 detections/frame
✅ Loaded 40567 LiDAR track-points
✅ Loaded 62266 Radar track-points
✅ Loaded 4399 Camera track-points
   Total: 107232 (compare to the original's millions — this is tracks, not raw points)


In [4]:
# CELL 4 - Within-frame sensor merging (the actual "fusion" step)
# If LiDAR and radar both see the same object this frame, merge them
# into one observation before matching against existing fused tracks.
#
# FIXED (Bug A): only merge observations from DIFFERENT sensors - two
# LiDAR (or two radar, or two camera) detections never get merged with
# each other, even if they are close. Two real pedestrians standing near
# each other, both seen only by LiDAR, must stay two separate detections.
#
# FIXED (Bug B): the old pass was greedy, not a real union-find - a chain
# of 3+ pairwise-close detections (A close to B, B close to C, A far from
# C) only grouped the first pair and left the third alone. This uses a
# proper disjoint-set union so the whole chain ends up in one cluster.
#
# FIXED (unweighted average): merged position used to be a plain mean of
# member positions, which treats a precise LiDAR reading and a noisier
# camera/radar reading as equally trustworthy. Now weighted by each
# sensor's approximate positional accuracy (SENSOR_TRUST_WEIGHT below),
# so the more accurate sensor dominates the merged estimate instead of
# being diluted by the noisier ones.

import numpy as np

# Inverse-variance-style trust weights from each sensor's approximate
# positional accuracy (LiDAR ~0.15m, radar sloppier laterally ~0.5m,
# camera-derived range least precise, especially for far objects ~1.0m).
# weight ~ 1 / sigma^2 -- the more accurate sensor dominates a merge.
SENSOR_TRUST_WEIGHT = {
    "lidar": 44.0,
    "radar": 4.0,
    "camera": 1.0,
}


def merge_frame_observations(observations, merge_thresh):
    """Union-find clustering: any chain of pairwise-close, DIFFERENT-sensor
    detections ends up in one cluster. Fine at this scale (tens of
    observations per frame, not thousands), so an O(n^2) edge pass is fast."""
    n = len(observations)
    if n == 0:
        return []

    positions = np.array([obs["pos"] for obs in observations])
    sensors = [obs["sensor"] for obs in observations]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sensors[i] == sensors[j]:
                continue  # never merge two detections from the same sensor
            if np.linalg.norm(positions[i] - positions[j]) < merge_thresh:
                union(i, j)

    clusters = {}
    for i in range(n):
        clusters.setdefault(find(i), []).append(i)

    merged = []
    for member_idx in clusters.values():
        members = [observations[i] for i in member_idx]
        member_pos = np.array([m["pos"] for m in members])
        weights = np.array([SENSOR_TRUST_WEIGHT.get(m["sensor"], 1.0) for m in members])
        avg_pos = (weights[:, None] * member_pos).sum(axis=0) / weights.sum()
        merged.append({
            "pos": avg_pos.tolist(),
            "sensors": sorted({m["sensor"] for m in members}),
            "source_track_ids": {m["sensor"]: m["source_track_id"] for m in members}
        })
    return merged


print("\u2705 merge_frame_observations() defined \u2014 cross-sensor only, union-find, trust-weighted.")

✅ merge_frame_observations() defined — cross-sensor only, union-find, trust-weighted.


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Fused tracker: one Hungarian assignment per frame + eviction
# ─────────────────────────────────────────────────────────────────

import uuid
from scipy.optimize import linear_sum_assignment


class FusedTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}
        self.finished_tracks = {}
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed

    def update(self, merged_observations, sample_id, timestamp):
        track_ids = list(self.active_tracks.keys())
        n_tracks, n_obs = len(track_ids), len(merged_observations)

        matched_track_idx, matched_obs_idx = set(), set()

        if n_tracks > 0 and n_obs > 0:
            # cost = np.zeros((n_tracks, n_obs))
            # for i, tid in enumerate(track_ids):
            #     last_pos = np.array(self.active_tracks[tid]["points"][-1]["pos"])
            #     for j, obs in enumerate(merged_observations):
            #         cost[i, j] = np.linalg.norm(np.array(obs["pos"]) - last_pos)

            cost = np.zeros((n_tracks, n_obs))
            for i, tid in enumerate(track_ids):
                pts = self.active_tracks[tid]["points"]       # [{"timestamp":..., "pos":[...]}, ...]
                last_pos = np.array(pts[-1]["pos"], dtype=float)
                pred = last_pos
                if len(pts) >= 2 and pts[-1]["timestamp"] is not None and pts[-2]["timestamp"] is not None:
                    dt_prev = (pts[-1]["timestamp"] - pts[-2]["timestamp"]) / 1e6
                    dt_now  = (timestamp              - pts[-1]["timestamp"]) / 1e6
                    if dt_prev > 0 and dt_now > 0:
                        vel = (last_pos - np.array(pts[-2]["pos"], dtype=float)) / dt_prev
                        spd = np.linalg.norm(vel)
                        if spd > 30.0:
                            vel = vel / spd * 30.0
                        pred = last_pos + vel * dt_now
                for j, obs in enumerate(merged_observations):
                    cost[i, j] = np.linalg.norm(np.array(obs["pos"], dtype=float) - pred)
            
            row_idx, col_idx = linear_sum_assignment(cost)
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < self.dist_thresh:
                    tid = track_ids[r]
                    obs = merged_observations[c]
                    self.active_tracks[tid]["points"].append({
                        "sample_id": sample_id, "timestamp": timestamp,
                        "pos": obs["pos"], "sensors": obs["sensors"]
                    })
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_obs_idx.add(c)

        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        for j in range(n_obs):
            if j not in matched_obs_idx:
                obs = merged_observations[j]
                tid = f"fused_{uuid.uuid4().hex[:8]}"
                self.active_tracks[tid] = {
                    "points": [{
                        "sample_id": sample_id, "timestamp": timestamp,
                        "pos": obs["pos"], "sensors": obs["sensors"]
                    }],
                    "missed": 0
                }

    def all_tracks(self):
        return {**self.finished_tracks, **self.active_tracks}


print("✅ FusedTracker defined.")

✅ FusedTracker defined.


In [6]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Main fusion loop
# ─────────────────────────────────────────────────────────────────

from tqdm import tqdm
import time

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

all_sample_ids = sorted(
    set(lidar_by_sample.keys()) | set(radar_by_sample.keys()) | set(camera_by_sample.keys())
)

tracker = FusedTracker(dist_thresh=FUSION_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)

start_time = time.time()
for sample_id in tqdm(all_sample_ids, desc="Fusing tracks"):
    frame_observations = (
        lidar_by_sample.get(sample_id, []) +
        radar_by_sample.get(sample_id, []) +
        camera_by_sample.get(sample_id, [])
    )

    merged = merge_frame_observations(frame_observations, INTRA_FRAME_MERGE_THRESH)

    timestamp = samples_index.get(sample_id, {}).get("timestamp", None)
    tracker.update(merged, sample_id, timestamp)

elapsed = time.time() - start_time
all_tracks = tracker.all_tracks()

print(f"\n✅ Fusion complete in {elapsed:.1f} seconds (was 22 hours, 4 minutes originally).")
print(f"   Total fused tracks: {len(all_tracks)}")

Fusing tracks:   0%|          | 0/404 [00:00<?, ?it/s]

Fusing tracks:   0%|          | 2/404 [00:00<00:40,  9.96it/s]

Fusing tracks:   1%|          | 3/404 [00:00<01:07,  5.94it/s]

Fusing tracks:   1%|          | 4/404 [00:00<01:36,  4.14it/s]

Fusing tracks:   1%|          | 5/404 [00:01<02:04,  3.21it/s]

...

Fusing tracks: 100%|█████████▉| 402/404 [01:58<00:00,  9.12it/s]

Fusing tracks: 100%|█████████▉| 403/404 [01:58<00:00,  9.23it/s]

Fusing tracks: 100%|██████████| 404/404 [01:58<00:00,  9.37it/s]

Fusing tracks: 100%|██████████| 404/404 [01:58<00:00,  3.40it/s]


✅ Fusion complete in 118.7 seconds (was 22 hours, 4 minutes originally).
   Total fused tracks: 18596


In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — Save: one combined CSV (primary, fast) + per-track JSON files
# (now feasible — thousands of files, not 1,057,292)
# ─────────────────────────────────────────────────────────────────

import pandas as pd

csv_rows = []
n_saved_json = 0

for tid, track in all_tracks.items():
    if len(track["points"]) < 2:
        continue

    for pt in track["points"]:
        csv_rows.append({
            "fused_id": tid,
            "sample_id": pt["sample_id"],
            "timestamp": pt["timestamp"],
            "x": pt["pos"][0], "y": pt["pos"][1], "z": pt["pos"][2] if len(pt["pos"]) > 2 else 0.0,
            "sensors": "+".join(pt["sensors"])
        })

    with open(FUSED_OUT_DIR / f"track_{tid}.json", "w") as f:
        json.dump(track["points"], f, indent=2)
    n_saved_json += 1

fused_df = pd.DataFrame(csv_rows)
csv_path = STEP4_DIR / "fused_tracks_all.csv"
fused_df.to_csv(csv_path, index=False)

print(f"✅ Combined CSV saved: {csv_path} ({len(fused_df)} rows)")
print(f"✅ {n_saved_json} individual track JSON files saved to: {FUSED_OUT_DIR}")

✅ Combined CSV saved: F:\Sensor fusion Research\output\step_4\fused_tracks_all.csv (81455 rows)
✅ 16589 individual track JSON files saved to: F:\Sensor fusion Research\output\step_4\fused


In [8]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — Summary: sensor contribution breakdown
# ─────────────────────────────────────────────────────────────────

multi_sensor_points = fused_df[fused_df["sensors"].str.contains(r"\+")]
single_sensor_points = fused_df[~fused_df["sensors"].str.contains(r"\+")]

print(f"Total fused track points     : {len(fused_df)}")
print(f"Points from 2+ sensors merged: {len(multi_sensor_points)} "
      f"({len(multi_sensor_points)/len(fused_df)*100:.1f}%)")
print(f"Points from a single sensor  : {len(single_sensor_points)}")

print("\nSensor combination breakdown:")
display(fused_df["sensors"].value_counts().head(10))

track_lengths = fused_df.groupby("fused_id").size()
summary_path = STEP4_DIR / "fusion_summary.csv"
track_lengths.reset_index(name="track_length").to_csv(summary_path, index=False)

print(f"\nMean fused track length: {track_lengths.mean():.1f} frames")
print(f"📄 Summary saved: {summary_path}")

Total fused track points     : 81455
Points from 2+ sensors merged: 5097 (6.3%)
Points from a single sensor  : 76358

Sensor combination breakdown:


sensors
radar                 48059
lidar                 27673
lidar+radar            2708
camera+lidar+radar     1152
camera+lidar            808
camera                  626
camera+radar            429
Name: count, dtype: int64


Mean fused track length: 4.9 frames
📄 Summary saved: F:\Sensor fusion Research\output\step_4\fusion_summary.csv


In [9]:
# Experiment: does loosening INTRA_FRAME_MERGE_THRESH increase multi-sensor merge rate
# without hurting merged-group accuracy?

def run_fusion_with_threshold(merge_thresh, dist_thresh=FUSION_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES):
    tracker = FusedTracker(dist_thresh=dist_thresh, max_missed=max_missed)
    for sample_id in all_sample_ids:
        frame_observations = (
            lidar_by_sample.get(sample_id, []) +
            radar_by_sample.get(sample_id, []) +
            camera_by_sample.get(sample_id, [])
        )
        merged = merge_frame_observations(frame_observations, merge_thresh)
        timestamp = samples_index.get(sample_id, {}).get("timestamp", None)
        tracker.update(merged, sample_id, timestamp)

    all_tracks = tracker.all_tracks()
    multi_sensor_count = sum(
        1 for t in all_tracks.values() for pt in t["points"] if len(pt["sensors"]) > 1
    )
    total_points = sum(len(t["points"]) for t in all_tracks.values())
    merge_rate = multi_sensor_count / total_points * 100 if total_points > 0 else 0
    return len(all_tracks), total_points, merge_rate


print(f"{'Threshold':<10} {'Tracks':<10} {'Points':<10} {'Merge Rate':<12}")
for thresh in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    n_tracks, n_points, merge_rate = run_fusion_with_threshold(thresh)
    marker = " ← current" if thresh == INTRA_FRAME_MERGE_THRESH else ""
    print(f"{thresh:<10} {n_tracks:<10} {n_points:<10} {merge_rate:<10.1f}%{marker}")

Threshold  Tracks     Points     Merge Rate  


1.5        19634      93513      5.0       %


2.0        19056      88556      5.7       %


2.5        18596      83462      6.2       % ← current


3.0        18201      78610      6.4       %


3.5        17851      73772      6.3       %


4.0        17497      69393      6.0       %
